# 1.6 Your turn: your own chat

Continues from [`01-cleaning.ipynb`](01-cleaning.ipynb) — same showcase pipeline,
pointed at your own export. If you do not have one yet, this notebook prints a
message and still finishes; come back to it once you do.

The one extra step is anonymisation. `humanize` maps each real name to a stable
nickname, so you can hand in your work without handing over your friends' names.

In [ ]:
from IPython.display import display

from goad_toolkit.datatransforms import Pipeline, RegexFeature, TimeFeatures

from wa_analyzer.data import PROCESSED, load_own_chat

`RegexFeature` is imported here, not rewritten. The class you derived in 1.4 is the
one `goad_toolkit` ships — same three modes, same `feature`-not-`name` parameter, same
coverage log on `extract`. Writing it once was the exercise; retyping it in every
notebook that needs it is just a copy waiting to drift out of step with the original.

In [ ]:
own = load_own_chat()

In [ ]:
if own is not None:
    from wa_analyzer.humanhasher import humanize

    anon = {name: humanize(name) for name in own.author.unique()}
    own["author"] = own.author.map(anon)
    print(f"{len(anon)} authors, anonymised. For example:")
    display(own.head(3))
else:
    print("No chat of your own yet — skipping. The showcase half above still stands.")

Now the *same pipeline*, pointed at a different dataframe. That is the payoff of having
written the steps as objects rather than as cells: nothing changes but the input.

In [ ]:
if own is not None:
    own_pipeline = Pipeline()
    own_pipeline.add(TimeFeatures, column="timestamp")
    own_pipeline.add(RegexFeature, name="urls",
                     column="message", pattern=r"https?://\S+",
                     feature="has_url", mode="has")
    own_pipeline.add(RegexFeature, name="questions",
                     column="message", pattern=r"\?",
                     feature="n_question", mode="count")

    own_enriched = own_pipeline.apply(own)
    display(own_enriched.head())

> **Your turn.** Write one `RegexFeature` step that is specific to *your* chat and would make
> no sense on IRC. Emoji, a group in-joke, a language you switch into, the way one person
> always signs off. Two of them.
>
> Chat data is unusually rich for this: a WhatsApp export gives you timestamps with daily and
> weekly rhythms, natural language, and half a dozen distributions with textbook shapes. Most
> of what makes this course work comes from features you extract yourself in this notebook.

In [ ]:
if own is not None:
    from datetime import datetime

    outfile = PROCESSED / f"chat-{datetime.now():%Y%m%d-%H%M%S}.parq"
    own_enriched.to_parquet(outfile, index=False)
    print(f"Wrote {outfile}")
    print("Put that filename after `current` in config.toml so the other notebooks find it.")

## 1.7 What to write down

Before moving on, you should be able to answer these about your own data. Not in your head —
written down, because lesson 2 starts by assuming them.

1. **What is one row, now?** One message. Say what a message is in your export — does a
   photo count, a system notice, a message that was deleted?
2. **How many rows did you lose, and to what?** Every parse drops something. Name the number
   and the reason.
3. **How many *people* are in your data?** Not messages — people. That number is much
   smaller, and in lesson 2 you will find out how much it matters.
4. **Which two features did you add, and what question is each for?** A feature you cannot
   attach to a question is one you will never use.
5. **What is in there that should not be?** Bots, group-admin notices, one person's phone
   posting twice. You do not have to remove them yet. You have to know they are there.

---

**Where this goes next.** The features you built here are the input to everything that
follows: lesson 2 compares them across people, lesson 3 across time, lesson 4 asks what shape
they have, and lesson 5 asks which of them tell people apart. And the habit — enrich before
you model — is the one the machine learning course builds on directly.